# Kvasir-VQA x1 — Fusion cross-attention baseline

Lightweight multimodal classifier: frozen ViT image encoder + frozen DistilBERT question encoder, fused via cross-attention and a small classifier on top-K answers.

In [ ]:
from pathlib import Path
import json
import random
from typing import Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import ViTImageProcessor, ViTModel
from transformers import DistilBertTokenizerFast, DistilBertModel

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "08_fusion_crossattn" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_MODEL_NAME = "google/vit-base-patch16-224-in21k"
TEXT_MODEL_NAME = "distilbert-base-uncased"
TOP_K_ANSWERS = 20
FREEZE_BACKBONES = True

BATCH_SIZE = 4
NUM_EPOCHS = 1
LR = 3e-4
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.01

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


In [ ]:
# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

# Clean
meta = meta.dropna(subset=["answer", "question", "image_path"]).reset_index(drop=True)
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

# Top-K answers
answer_counts = meta["answer"].value_counts()
top_answers = set(answer_counts.head(TOP_K_ANSWERS).index)
meta = meta[meta["answer"].isin(top_answers)].reset_index(drop=True)

answer_to_id = {ans: i for i, ans in enumerate(sorted(top_answers))}
id_to_answer = {v: k for k, v in answer_to_id.items()}
meta["label_id"] = meta["answer"].map(answer_to_id)

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df   = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df  = meta[meta["split"] == "test"].reset_index(drop=True)

if MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


In [ ]:
# Datasets
image_processor = ViTImageProcessor.from_pretrained(IMAGE_MODEL_NAME)
text_tokenizer = DistilBertTokenizerFast.from_pretrained(TEXT_MODEL_NAME)

class VQADS(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.paths = df["image_path"].tolist()
        self.questions = df["question"].tolist()
        self.labels = df["label_id"].astype(int).tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        image_inputs = image_processor(images=img, return_tensors="pt")
        text_inputs = text_tokenizer(
            self.questions[idx],
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt",
        )
        item = {
            "pixel_values": image_inputs["pixel_values"].squeeze(0),
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }
        return item

def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_ds = VQADS(train_df)
val_ds   = VQADS(val_df)
test_ds  = VQADS(test_df)


In [ ]:
# Fusion model
class FusionModel(nn.Module):
    def __init__(self, num_labels: int, freeze_backbones: bool = True):
        super().__init__()
        self.text_encoder = DistilBertModel.from_pretrained(TEXT_MODEL_NAME)
        self.image_encoder = ViTModel.from_pretrained(IMAGE_MODEL_NAME)

        text_hidden = self.text_encoder.config.hidden_size
        img_hidden = self.image_encoder.config.hidden_size

        self.img_proj = nn.Linear(img_hidden, text_hidden)
        self.cross_attn = nn.MultiheadAttention(embed_dim=text_hidden, num_heads=4, batch_first=True)
        self.classifier = nn.Linear(text_hidden, num_labels)

        if freeze_backbones:
            for p in self.text_encoder.parameters():
                p.requires_grad = False
            for p in self.image_encoder.parameters():
                p.requires_grad = False

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        img_out = self.image_encoder(pixel_values=pixel_values)
        img_tokens = self.img_proj(img_out.last_hidden_state)  # (B, img_seq, text_hidden)

        txt_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        txt_tokens = txt_out.last_hidden_state  # (B, txt_seq, text_hidden)

        attn_output, _ = self.cross_attn(query=txt_tokens, key=img_tokens, value=img_tokens)
        # Masked mean pool over text tokens (after cross-attn)
        mask = attention_mask.unsqueeze(-1)  # (B, seq, 1)
        attn_output = attn_output * mask
        pooled = attn_output.sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return logits, loss

model = FusionModel(num_labels=len(answer_to_id), freeze_backbones=FREEZE_BACKBONES).to(DEVICE)
print(model.classifier)


In [ ]:
# Train / eval loops
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_f1 = 0


def run_epoch(loader, train_mode: bool):
    model.train(mode=train_mode)
    total_loss = 0
    preds = []
    labels_all = []
    for batch in tqdm(loader, leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        logits, loss = model(**batch)
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(dim=-1).detach().cpu().tolist())
        labels_all.extend(batch["labels"].detach().cpu().tolist())
    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(labels_all, preds) if preds else 0
    f1 = f1_score(labels_all, preds, average="macro") if preds else 0
    return avg_loss, acc, f1

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_f1 = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc, val_f1 = run_epoch(val_loader, train_mode=False)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUT_DIR / "best_model.pt")

# Load best and evaluate on test
if (OUT_DIR / "best_model.pt").exists():
    model.load_state_dict(torch.load(OUT_DIR / "best_model.pt", map_location=DEVICE))

_, test_acc, test_f1 = run_epoch(test_loader, train_mode=False)
metrics = {
    "val": {"macro_f1": best_val_f1},
    "test": {"accuracy": test_acc, "macro_f1": test_f1},
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)
